# Lab: HR Employee Dataset — Data Preparation for Analysis

In [1]:
import pandas as pd
import numpy as np

# Do not modify this cell.
# Raw HR employee dataset created for data cleaning practice.

employees = pd.DataFrame({
    "EmployeeID": [
        101, 102, 103, 103, 105, 106, 107, 108,
        109, 110, 111, 112, 113, 114, 115
    ],
    "FirstName": [
        "Ainura", "Bakyt", "Gulnara", "Gulnara", "Nurlan",
        "Aigul", "Beksultan", "Aisuluu", "Ermek", "Nurgul",
        "Daniyar", "Aizhan", "Kubat", "Perizat", "Adilet"
    ],
    "LastName": [
        "Asanova", "Kyzylov", "Mamatova", "Mamatova", "Abdyldaev",
        "Sultanova", "Toktogulov", "baisalova", "Kadyrov",
        "Jumabaeva ", "Turgunov", "Ryskulova", "Kadyraliev",
        "BAktybekova", "MOLDOKULOV"
    ],
    "Department": [
        "IT", "HR", "it", "IT", "Finance", "HR",
        " IT ", "Finance", "IT", "HR", "it",
        "Financing", "Information Tecnology", "HR", "finance"
    ],
    "Gender": [
        "F", "m", "F", "f", "M", "F", "M", "F",
        "M", "F", "M", "F", "M", "F", "m"
    ],
    "Age": [
        "28", "35", "29", "29", "150", "31", "33",
        "27", "30", "38", "26", "34", "29", "32", "31"
    ],
    "Salary": [
        "80000", "65000", np.nan, np.nan, "90000", "70000",
        "unknown", "62000", "85000", "58000", "72000",
        "66000", "78000", "88000", "61000"
    ],
    "HireDate": [
        "2023-01-15", "2022-06-10", "2024-03-01", "2024-03-01",
        "2021-11-20", "2023-08-05", "2020-05-12", "2022-09-18",
        "2021-03-25", "2019-07-14", "2023-04-18", "2020-10-02",
        "2022-12-11", "2021-08-30", "2023-06-22"
    ],
    "PerformanceScore": [
        4.2, 3.8, 4.5, 4.5, 4.0, 3.9, 4.1,
        3.7, 4.3, 3.6, 4.4, 3.8, 4.2, "4.0", 3.9
    ]
})

departments = pd.DataFrame({
    "DepartmentID": [1, 2, 3],
    "DepartmentName": ["IT", "HR", "Finance"],
    "Location": ["Bishkek", "Osh", "Bishkek"]
})

In [2]:
df = employees.copy()

## Task 1 — Inspect the Dataset

In [3]:
print("Rows, columns:", df.shape)
print()
df.info()
print()
print(df[["Salary", "Age", "HireDate"]].dtypes)

Rows, columns: (15, 9)

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   EmployeeID        15 non-null     int64 
 1   FirstName         15 non-null     str   
 2   LastName          15 non-null     str   
 3   Department        15 non-null     str   
 4   Gender            15 non-null     str   
 5   Age               15 non-null     str   
 6   Salary            13 non-null     str   
 7   HireDate          15 non-null     str   
 8   PerformanceScore  15 non-null     object
dtypes: int64(1), object(1), str(7)
memory usage: 1.2+ KB

Salary      str
Age         str
HireDate    str
dtype: object


In [4]:
print("Rows where Salary is present (notna):", df["Salary"].notna().sum())
print()
print("Missing values:")
print(df.isna().sum())
print()
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate EmployeeID:", df["EmployeeID"].duplicated().sum())
print()
print("Departments:", df["Department"].unique())
print("Genders:", df["Gender"].unique())
print()
print("Age above 100:")
print(df[df["Age"].astype(int) > 100][["EmployeeID", "FirstName", "Age"]])
print()
print("Types in PerformanceScore:")
print(df["PerformanceScore"].map(type).value_counts())

Rows where Salary is present (notna): 13

Missing values:
EmployeeID          0
FirstName           0
LastName            0
Department          0
Gender              0
Age                 0
Salary              2
HireDate            0
PerformanceScore    0
dtype: int64

Exact duplicate rows: 0
Duplicate EmployeeID: 1

Departments: <StringArray>
[                   'IT',                    'HR',                    'it',
               'Finance',                  ' IT ',             'Financing',
 'Information Tecnology',               'finance']
Length: 8, dtype: str
Genders: <StringArray>
['F', 'm', 'f', 'M']
Length: 4, dtype: str

Age above 100:
   EmployeeID FirstName  Age
4         105    Nurlan  150

Types in PerformanceScore:
PerformanceScore
<class 'float'>    14
<class 'str'>       1
Name: count, dtype: int64


In [5]:
text_cols = ["FirstName", "LastName", "Department", "Gender"]
df[text_cols] = df[text_cols].apply(lambda col: col.str.strip())

print(df.loc[[6, 9], ["LastName", "Department"]])

     LastName Department
6  Toktogulov         IT
9   Jumabaeva         HR


Проблемы: пропуски и `unknown` в Salary, дубликат EmployeeID 103, разные названия отделов, Age = 150, числа и даты в виде текста, строка в PerformanceScore.

## Task 2 — Handle Missing Salaries

In [6]:
missing_count = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(1)
print(pd.DataFrame({"missing": missing_count, "percent": missing_percent}))

                  missing  percent
EmployeeID              0      0.0
FirstName               0      0.0
LastName                0      0.0
Department              0      0.0
Gender                  0      0.0
Age                     0      0.0
Salary                  2     13.3
HireDate                0      0.0
PerformanceScore        0      0.0


In [7]:
df["Salary"] = df["Salary"].replace("unknown", np.nan)
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")

print(df["Salary"].dtype)
print("Missing Salary:", df["Salary"].isna().sum())

float64
Missing Salary: 3


In [8]:
print("original shape:                 ", df.shape)
print("dropna():                       ", df.dropna().shape)
print("dropna(axis=1):                 ", df.dropna(axis=1).shape)
print("dropna(how='all'):              ", df.dropna(how="all").shape)
print("dropna(subset=['Salary']):      ", df.dropna(subset=["Salary"]).shape)
print("dropna(subset=['Age']):         ", df.dropna(subset=["Age"]).shape)
print("dropna(thresh=9):               ", df.dropna(thresh=9).shape)
print("dropna(thresh=8):               ", df.dropna(thresh=8).shape)

original shape:                  (15, 9)
dropna():                        (12, 9)
dropna(axis=1):                  (15, 8)
dropna(how='all'):               (15, 9)
dropna(subset=['Salary']):       (12, 9)
dropna(subset=['Age']):          (15, 9)
dropna(thresh=9):                (12, 9)
dropna(thresh=8):                (15, 9)


In [9]:
median_salary = df["Salary"].median()
df["Salary"] = df["Salary"].fillna(median_salary)

print("Median salary:", median_salary)
print("Missing Salary after fillna:", df["Salary"].isna().sum())

Median salary: 71000.0
Missing Salary after fillna: 0


Пропуски заполнил медианой (71 000). `dropna`: по умолчанию убирает строки с любым пропуском, `axis=1` — столбцы, `how="all"` — только полностью пустые строки, `subset` — смотрит выбранные столбцы, `thresh` — минимум непустых значений.

## Task 3 — Standardize Department Names

In [10]:
df["Department"] = (
    df["Department"]
    .str.strip()
    .str.lower()
    .str.replace("information tecnology", "it", regex=False)
    .str.replace("financing", "finance", regex=False)
    .map({"it": "IT", "hr": "HR", "finance": "Finance"})
)

df["Gender"] = df["Gender"].str.upper()
df["FirstName"] = df["FirstName"].str.title()
df["LastName"] = df["LastName"].str.title()

print(df["Department"].unique())
print(df["Department"].value_counts())

<StringArray>
['IT', 'HR', 'Finance']
Length: 3, dtype: str
Department
IT         7
HR         4
Finance    4
Name: count, dtype: int64


In [11]:
age_number = pd.to_numeric(df["Age"], errors="coerce")
print(df[(age_number < 18) | (age_number > 70)][["EmployeeID", "FirstName", "LastName", "Age"]])

df["Age"] = age_number.where(age_number.between(18, 70))
print("Missing Age:", df["Age"].isna().sum())

   EmployeeID FirstName   LastName  Age
4         105    Nurlan  Abdyldaev  150
Missing Age: 1


Отделы привёл к IT, HR, Finance. Возраст 150 (сотрудник 105) заменил на NaN.

## Task 4 — Remove Duplicates

In [12]:
print("Exact duplicates:", df.duplicated().sum())
print(df[df.duplicated(keep=False)])

Exact duplicates: 1
   EmployeeID FirstName  LastName Department Gender   Age   Salary  \
2         103   Gulnara  Mamatova         IT      F  29.0  71000.0   
3         103   Gulnara  Mamatova         IT      F  29.0  71000.0   

     HireDate PerformanceScore  
2  2024-03-01              4.5  
3  2024-03-01              4.5  


In [13]:
print("subset=['EmployeeID'], keep='first':", df.drop_duplicates(subset=["EmployeeID"], keep="first").shape[0])
print("subset=['EmployeeID'], keep='last': ", df.drop_duplicates(subset=["EmployeeID"], keep="last").shape[0])
print("subset=['EmployeeID'], keep=False:  ", df.drop_duplicates(subset=["EmployeeID"], keep=False).shape[0])

subset=['EmployeeID'], keep='first': 14
subset=['EmployeeID'], keep='last':  14
subset=['EmployeeID'], keep=False:   13


In [14]:
df = df.drop_duplicates()
print("After exact duplicates:", df.shape[0])

df = df.drop_duplicates(subset=["EmployeeID"], keep="first")
print("After duplicates by EmployeeID:", df.shape[0])

print("Duplicate EmployeeID left:", df["EmployeeID"].duplicated().sum())

After exact duplicates: 14
After duplicates by EmployeeID: 14
Duplicate EmployeeID left: 0


После очистки строка сотрудника 103 стала точным дубликатом, удалил её. `keep=False` удаляет все копии.

## Task 5 — Convert Data Types

In [15]:
print("Before:")
print(df[["Age", "Department", "Gender", "PerformanceScore"]].dtypes)
print()

memory_before = df["Department"].memory_usage(deep=True)

df["Age"] = df["Age"].astype("Int64")
print("Age:", df["Age"].dtype)

df["Department"] = df["Department"].astype("category")
print("Department:", df["Department"].dtype)

df["Gender"] = df["Gender"].astype("category")
print("Gender:", df["Gender"].dtype)

df["PerformanceScore"] = pd.to_numeric(df["PerformanceScore"], errors="coerce")
print("PerformanceScore:", df["PerformanceScore"].dtype)

print()
print("Department memory:", memory_before, "->", df["Department"].memory_usage(deep=True), "bytes")
print("Categories:", list(df["Department"].cat.categories))

Before:
Age                 float64
Department              str
Gender                  str
PerformanceScore     object
dtype: object

Age: Int64
Department: category
Gender: category
PerformanceScore: float64

Department memory: 846 -> 284 bytes
Categories: ['Finance', 'HR', 'IT']


`category` хранит уникальные значения один раз, поэтому занимает меньше памяти, чем `object`.

## Task 6 — Parse Hire Dates

In [16]:
df["HireDate"] = pd.to_datetime(df["HireDate"], format="%Y-%m-%d")

df["HireYear"] = df["HireDate"].dt.year
df["HireMonth"] = df["HireDate"].dt.month
df["HireWeekday"] = df["HireDate"].dt.day_name()

reference_date = pd.Timestamp("2026-10-27")
df["YearsWorked"] = ((reference_date - df["HireDate"]).dt.days / 365.25).round(1)

print(df["HireDate"].dtype)
print(df[["EmployeeID", "HireDate", "HireYear", "HireMonth", "HireWeekday", "YearsWorked"]])

datetime64[us]
    EmployeeID   HireDate  HireYear  HireMonth HireWeekday  YearsWorked
0          101 2023-01-15      2023          1      Sunday          3.8
1          102 2022-06-10      2022          6      Friday          4.4
2          103 2024-03-01      2024          3      Friday          2.7
4          105 2021-11-20      2021         11    Saturday          4.9
5          106 2023-08-05      2023          8    Saturday          3.2
6          107 2020-05-12      2020          5     Tuesday          6.5
7          108 2022-09-18      2022          9      Sunday          4.1
8          109 2021-03-25      2021          3    Thursday          5.6
9          110 2019-07-14      2019          7      Sunday          7.3
10         111 2023-04-18      2023          4     Tuesday          3.5
11         112 2020-10-02      2020         10      Friday          6.1
12         113 2022-12-11      2022         12      Sunday          3.9
13         114 2021-08-30      2021          8   

Даты переведены в datetime, добавлены год, месяц, день недели и стаж.

## Task 7 — Rename Columns and Merge Datasets

In [17]:
df = df.rename(columns={"Salary": "MonthlySalary", "HireDate": "EmploymentStartDate"})
print(df.columns.tolist())

['EmployeeID', 'FirstName', 'LastName', 'Department', 'Gender', 'Age', 'MonthlySalary', 'EmploymentStartDate', 'PerformanceScore', 'HireYear', 'HireMonth', 'HireWeekday', 'YearsWorked']


In [18]:
merged = pd.merge(
    df,
    departments,
    how="left",
    left_on="Department",
    right_on="DepartmentName",
)

merged["Department"] = merged["Department"].astype("category")

print("Rows before / after merge:", df.shape[0], "/", merged.shape[0])
print("Missing after merge:")
print(merged[["DepartmentID", "DepartmentName", "Location"]].isna().sum())
print()
print(merged[["EmployeeID", "Department", "DepartmentID", "Location"]])

Rows before / after merge: 14 / 14
Missing after merge:
DepartmentID      0
DepartmentName    0
Location          0
dtype: int64

    EmployeeID Department  DepartmentID Location
0          101         IT             1  Bishkek
1          102         HR             2      Osh
2          103         IT             1  Bishkek
3          105    Finance             3  Bishkek
4          106         HR             2      Osh
5          107         IT             1  Bishkek
6          108    Finance             3  Bishkek
7          109         IT             1  Bishkek
8          110         HR             2      Osh
9          111         IT             1  Bishkek
10         112    Finance             3  Bishkek
11         113         IT             1  Bishkek
12         114         HR             2      Osh
13         115    Finance             3  Bishkek


Объединил с departments по названию отдела, пропусков после merge нет.

## Task 8 — Final Check and Summary

In [19]:
print("Shape:", merged.shape)
print()
print(merged.dtypes)
print()
print("Missing values:")
print(merged.isna().sum())
print()
print("Duplicate rows:", merged.duplicated().sum())
print("Duplicate EmployeeID:", merged["EmployeeID"].duplicated().sum())
print("Departments:", merged["Department"].unique().tolist())
print("Date column type:", merged["EmploymentStartDate"].dtype)

Shape: (14, 16)

EmployeeID                      int64
FirstName                         str
LastName                          str
Department                   category
Gender                       category
Age                             Int64
MonthlySalary                 float64
EmploymentStartDate    datetime64[us]
PerformanceScore              float64
HireYear                        int32
HireMonth                       int32
HireWeekday                       str
YearsWorked                   float64
DepartmentID                    int64
DepartmentName                    str
Location                          str
dtype: object

Missing values:
EmployeeID             0
FirstName              0
LastName               0
Department             0
Gender                 0
Age                    1
MonthlySalary          0
EmploymentStartDate    0
PerformanceScore       0
HireYear               0
HireMonth              0
HireWeekday            0
YearsWorked            0
DepartmentID       